# Assignment 3: Summarization Tests

**Description:** This assignment covers summarization outputs. You will compare three different types of solutions, all using an encoder-decoder architecture. You should also be able to develop an intuition for:


* How well summarization systems work
* The effects of using different pre-training and fine-tuning checkpoints on outcomes
* The effects of hyperparameters on outcomes
* Evaluation of output using ROUGE



This notebook should be run on a Google Colab but it does not require a GPU. By default, when you open the notebook in Colab it will NOT configure a GPU.  Summarization commands can take up to five minutes to run depending on the hyperparameters you use. This notebook will NOT run on your GCP instance as the summary models are larger than the avaialble memory.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/datasci-w266/2023-summer-main/blob/master/assignment/a3/Summarization_test.ipynb)

The overall assignment structure is as follows:

 Setup

1. T5 for generic summarization

2. Pegasus for headline summarization

3. Pegasus for longer generation




**INSTRUCTIONS:**:

* Questions are always indicated as **QUESTION:**, so you can search for this string to make sure you answered all of the questions. You are expected to fill out, run, and submit this notebook, as well as to answer the questions in the **answers** file as you did in a1 and a2.

* **### YOUR CODE HERE** indicates that you are supposed to write code.




## Setup

In [1]:
!pip install -q sentencepiece

In [2]:
!pip install -q transformers

In [3]:
!pip install -q evaluate
import evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00


In [4]:
!pip install -q rouge_score

  Preparing metadata (setup.py) ... done


In [5]:
#let's make longer output readable without horizontal scrolling
from pprint import pprint

Let's leverage the pre-trained and fine tuned models on HuggingFace to demonstrate some capabilities with abstractive summarization and language generation.  They include models/checkpoints that were fine tuned on a particular dataset.  In our case we'll focus on one dataset that emphasizes a one line output and another that emphasizes a multi-line output.

We'll use this same toy article as the input to all of our summarization attempts.  That way we have the ability to compare. We'll also create two references for evaluation.  These are the targets you are trying to meet.  One reference is for the longer output. The second reference is the short one for the one line output.

In [6]:

ARTICLE_TO_SUMMARIZE = (
    "Nearly 800 thousand customers are scheduled to be affected by the shutoffs which are expected to last through at least midday tomorrow. "
    "PG&E stated it scheduled the blackouts in response to forecasts for high winds amid dry conditions. "
    "The aim is to reduce the risk of wildfires. "
    "If Pacific Gas & Electric Co, a unit of PG&E Corp, goes through with another public safety power shutoff, "
    " it would be the fourth round of mass blackouts imposed by the utility since Oct. 9, when some 730,000 customers were left in the dark. "
    "The recent wave of precautionary shutoffs have drawn sharp criticism from Governor Gavin Newsom, state regulators and consumer activists as being overly broad in scale."
    "Newsom blames PG&E for doing too little to properly maintain and secure its power lines against wind damage."
    "Utility executives have acknowledged room for improvement while defending the sprawling scope of the power cutoffs as a matter of public safety."
    "The record breaking drought has made the current conditions even worse than in previous years. "
    "It exponentially increases the probability of large scale wildfires. "
)

LONG_REFERENCE = (
    "Many PG&E customers could be affected by public safety power shutoffs in response to forecasts for high winds and dry conditions. "
    "The record breaking drought exponentially increases the probability of large scale wildfires. "
    "Despite being criticized by Governor Newsom for being overly broad, company officials defend the cutoffs as a matter of public safety. "
)

SHORT_REFERENCE = (
    "California's largest utility is set to turn off power to hundreds of thousands of customers in an effort to reduce the risk of wildfires. "
)

How long is our article to summarize?  Obviously our summary should be shorter since it is supposed to be "abridged."

In [7]:
len(str.split(ARTICLE_TO_SUMMARIZE))

177

## 1. T5 for Generic Summarization

T5 is an encoder decoder architecture that has been trained on multiple tasks, so not purely summarization.  You can read more about it [here](https://huggingface.co/docs/transformers/model_doc/t5).

In [8]:
from transformers import T5Tokenizer, TFT5ForConditionalGeneration

t5model = TFT5ForConditionalGeneration.from_pretrained("t5-base")
t5tokenizer = T5Tokenizer.from_pretrained("t5-base")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

All PyTorch model weights were used when initializing TFT5ForConditionalGeneration.

All the weights of TFT5ForConditionalGeneration were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFT5ForConditionalGeneration for predictions without further training.


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [9]:
t5model.summary()

Model: "tft5_for_conditional_generation"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 shared (Embedding)          multiple                  24674304  
                                                                 
 encoder (TFT5MainLayer)     multiple                  109628544 
                                                                 
 decoder (TFT5MainLayer)     multiple                  137949312 
                                                                 
Total params: 222903552 (850.31 MB)
Trainable params: 222903552 (850.31 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


Since T5 can perform multiple tasks we need to tell it what kind of output we want.  Therefore we need to prepend a "prompt" to our article text to make sure it does the right thing.

In [10]:
PROMPT = 'summarize: '
T5ARTICLE_TO_SUMMARIZE = PROMPT + ARTICLE_TO_SUMMARIZE

In [11]:
inputs = t5tokenizer(T5ARTICLE_TO_SUMMARIZE, max_length=1024, truncation=True, return_tensors="tf")

What do the inputs look like?  How does it compare with what we've seen from BERT?

In [12]:
inputs

{'input_ids': <tf.Tensor: shape=(1, 241), dtype=int32, numpy=
array([[21603,    10, 10455,   120,  8640,  7863,   722,    33,  5018,
           12,    36,  4161,    57,     8,  6979,  1647,     7,    84,
           33,  1644,    12,   336,   190,    44,   709,  2076,  1135,
         5721,     5,     3,  7861,   184,   427,  4568,    34,  5018,
            8,  1001,   670,     7,    16,  1773,    12,  7555,     7,
           21,   306, 13551, 18905,  2192,  1124,     5,    37,  2674,
           19,    12,  1428,     8,  1020,    13,  3645,  6608,     7,
            5,   156,  5824,  6435,     3,   184,  8666,   638,     6,
            3,     9,  1745,    13,     3,  7861,   184,   427, 10052,
            6,  1550,   190,    28,   430,   452,  1455,   579,  6979,
         1647,     6,    34,   133,    36,     8,  4509,  1751,    13,
         3294,  1001,   670,     7,     3, 16068,    57,     8,  6637,
          437,  6416,     5,  9902,   116,   128,   489, 17093,   722,
          130, 

Let's just run T5 using it's default hyperparameters and see what happens.  We'll hold on to the output in the candidate variable.  What do you think about the output?

In [13]:
# Generate Summary
summary_ids = t5model.generate(inputs["input_ids"],
                               max_new_tokens=30
)
candidate = t5tokenizer.batch_decode(summary_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
pprint(candidate[0], compact=True)

('PG&e shuts off power to 800 thousand customers . the shutoffs are scheduled '
 'to last through at least midday tomorrow .')


It doesn't do too bad of a job honestly, but it still could be more coherent.

### 1.a Checkpoint Configuration

We're using the `t5-base` configuration and we know we can run out of the box to do summarization which means it has some hyperparameters set as defaults.  These may or may not be what we want to use.  How do we know which values are set as defaults?

HuggingFace provides access to the default hyperparameters via the AutoConfig object which we call below.  We simply pass in the name of the checkpoint we're using -- `t5-base` in this case.


In [14]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("t5-base")

config

T5Config {
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 3072,
  "d_kv": 64,
  "d_model": 768,
  "decoder_start_token_id": 0,
  "dense_act_fn": "relu",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "relu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": false,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 12,
  "num_heads": 12,
  "num_layers": 12,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "task_specific_params": {
    "summarization": {
      "early_stopping": true,
      "length_penalty": 2.0,
      "max_length": 200,
      "min_length": 30,
      "no_repeat_ngram_size": 3,
      "num_beams": 4,
      "prefix": "summarize: "
    },
    "translation_en_to_de": {
      "early_stopping": true,
      "max_length": 300,
      "num_beams": 4,
      "pref

Look at the `task_specific_params` for summarization. You can see that this `t5-base` checkpoint has some values such as min_length and max_length as well as no_repeat_ngram_size and num_beams.  You can affect the size and content of the output by modifying these parameters which you will do below.

You can also look at the full set of possible parameters in the [TFGenerationMixin](https://huggingface.co/docs/transformers/v4.18.0/en/main_classes/text_generation#transformers.generation_tf_utils.TFGenerationMixin) class available to all of the pre-trained models.

HuggingFace has also written [a very helpful blog post](https://huggingface.co/blog/how-to-generate) that explains and discusses various strategies for text generation and how to manipulate the hyperparameters.  They discuss the two approaches of beam search (which we have discussed in the async and live session) as well as sampling (which tries to randomly pick the next word within a k-sized distribution of highly probable choices).

**Please read the blog post before you proceed.**

For your reference, here's a more complex, technical, and thorough [HuggingFace guide](https://huggingface.co/docs/transformers/main/en/generation_strategies) for controlling generation of text.  The blog post above is all you need to read to complete the assignment.

### 1.b ROUGE for summarization evaluation

ROUGE is the metric that has been traditionally used to evaluate sumarization results.  The ROUGE metric expects a reference as input and it will evaluate a candidate against that reference.  ROUGE-1 calculates the number of words in the reference that occur in the candidate.  ROUGE-2 performs that same calculation but for bigrams in the reference. ROUGE-L calculates the longest common subsequence of reference words that occur in the candidate.

HuggingFace provides a wrapper around [a library](https://huggingface.co/spaces/evaluate-metric/rouge) to calculate ROUGE metrics which you will use below.  Let's calculate the ROUGE score for the candidate you produced above.

In [15]:
rouge = evaluate.load('rouge')
predictions = candidate
references = [SHORT_REFERENCE]
results = rouge.compute(predictions=predictions,
                        references=references)
print(results)

{'rouge1': np.float64(0.2666666666666666), 'rouge2': np.float64(0.0930232558139535), 'rougeL': np.float64(0.22222222222222224), 'rougeLsum': np.float64(0.22222222222222224)}


ROUGE-L ignores newlines and computes the LCS (longest common subsequence) for the entire text. This is appropriate when the candidate and reference summaries contain a single long sequence. ROUGE-Lsum splits the text into sentences based on newlines and computes the LCS for each pair of sentences. It then takes the union of all LCS scores (which in our case, is the same).

Let's experiment with the hyperparameters shown above.  Please experiment in the cell below.  The `num_beams` value is like a beam search.  It indicates the number of tries the model makes before showing you its best output.  The `no_repeat_ngram_size` is designed to help reduce repetition in the output.  `min_length` and `max_length` (or now `max_new_tokens`) set boundaries on the size of the summary. You are free to use other hyperparameters as described in the [blog post](https://huggingface.co/blog/how-to-generate).

*There is no one correct answer to these questions.  There are ranges that tend to work better than others.  The goal is to have you experiment to help build intuition.  Please enter the values that you think are generating the most readable output.*

*Your readable output should consist of at least one complete sentence but does not have to end with a period and you must also have a ROUGE-1 score above 0.30 and ROUGE-L score equal to or above 0.25 when compared with the short reference.*

You can use the two cells below to come up with your answer.

In [16]:
SHORT_REFERENCE

"California's largest utility is set to turn off power to hundreds of thousands of customers in an effort to reduce the risk of wildfires. "

In [ ]:
# Generate Summary
summary_ids = t5model.generate(inputs["input_ids"],
### YOUR CODE HERE
                               max_new_tokens=40
### END YOUR CODE
)

candidate = t5tokenizer.batch_decode(summary_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
pprint(candidate[0], compact=True)

('PG&e shuts off power to 800 thousand customers . the shutoffs are scheduled '
 'to last through at least midday tomorrow . the aim is to reduce the risk of '
 'wildfire')


In [ ]:
predictions = candidate
references = [SHORT_REFERENCE]
results = rouge.compute(predictions=predictions,
                        references=references)
print(results)

{'rouge1': np.float64(0.4074074074074074), 'rouge2': np.float64(0.23076923076923075), 'rougeL': np.float64(0.3333333333333333), 'rougeLsum': np.float64(0.3333333333333333)}


The following code I will use to alter each hyperparameter individually, just to get a feel of the sentence and what it produces. I will loop through the increasing levels of the altered hyperparameter 3 times, just so I can get a rough average.

In [17]:
import pandas as pd
import numpy as np
from tqdm import tqdm

In [18]:
# Defining helper function for hyperparameter testing, looping hyperparameters, and displaying results
def _hp_testing(model, tokenizer, input, short_reference, max_new_tokens=10,
                num_beams=1, early_stopping=False, no_repeat_ngram_size=0,
                do_sample=False, top_k=50, temperature=1, top_p=1,
                min_length=10):

  # Initializing final list to feed into final df later on
  final_list = []

  # Generating summary_ids (initialized with default settings)
  summary_ids = model.generate(input,
                               max_new_tokens=max_new_tokens,
                               num_beams=num_beams,
                               early_stopping=early_stopping,
                               no_repeat_ngram_size=no_repeat_ngram_size,
                               do_sample=do_sample,
                               top_k=top_k,
                               temperature=temperature,
                               top_p=top_p,
                               min_length=min_length)

  # Generating the candidate
  candidate = tokenizer.batch_decode(summary_ids,
                                     skip_special_tokens=True,
                                     clean_up_tokenization_spaces=False)
  predictions = candidate

  # Getting the results of the predictions
  results = rouge.compute(predictions=predictions,
                          references=[short_reference])

  # Appending the rouge scores along w/ the generated sentence
  final_list.append(results['rouge1'])
  final_list.append(results['rouge2'])
  final_list.append(results['rougeL'])
  final_list.append(results['rougeLsum'])
  final_list.append(candidate[0])

  return final_list

def loop_through_hp(model, tokenizer, input, short_reference, changed_hp,
  range_start, range_stop, step, max_new_tokens=40, num_beams=1,
  early_stopping=False, no_repeat_ngram_size=0, do_sample=False, top_k=50,
  temperature=1, top_p=1, min_length=10):

  # Initialize final df
  final_df = pd.DataFrame(columns=['hp_value', 'rouge1', 'rouge2', 'rougeL',
                                   'rougeLsum', 'sent_gen'])

  # Looping through the range of hp values to test
  for i in tqdm(range(range_start, range_stop, step)):

    # Changing the updated hp based on what's chosen:
    if changed_hp == 'max_new_tokens':
      max_new_tokens = i
    elif changed_hp == 'num_beams':
      num_beams = i
    elif changed_hp == 'early_stopping':
      early_stopping = i
    elif changed_hp == 'no_repeat_ngram_size':
      no_repeat_ngram_size = i
    elif changed_hp == 'do_sample':
      do_sample = i
    elif changed_hp == 'top_k':
      top_k = i
    elif changed_hp == 'temperature':
      temperature = i/10.0
    elif changed_hp == 'top_p':
      top_p = i/100.0
    elif changed_hp == 'min_length':
      min_length = i

    # Getting the list results from the previous function
    list_results = _hp_testing(model, tokenizer, input, short_reference,
                               max_new_tokens, num_beams, early_stopping,
                               no_repeat_ngram_size, do_sample, top_k,
                               temperature, top_p, min_length)
    list_results.insert(0, i)

    # Updating final df
    final_df.loc[len(final_df)] = list_results

  return final_df

def display_df_and_sent(df):
  display(df)
  for i in range(len(df)):
    print('\nSentence for', changed_hp, '=', df['hp_value'].iloc[i])
    pprint(df['sent_gen'].iloc[i], compact=True)

Testing max_new_tokens from 20 to 60:

In [ ]:
# Updating hyperparameter to change
changed_hp = 'max_new_tokens'
range_start = 20
range_stop = 61
step = 5

# Getting the df
df_1 = loop_through_hp(t5model, t5tokenizer, inputs["input_ids"],
  SHORT_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=40,
  num_beams=1, early_stopping=False, no_repeat_ngram_size=0, do_sample=False,
  top_k=50, temperature=1, top_p=1)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_1)

100%|██████████| 9/9 [01:58<00:00, 13.12s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,20,0.263158,0.111111,0.263158,0.263158,PG&e shuts off power to 800 thousand customers...
1,25,0.279070,0.097561,0.232558,0.232558,PG&e shuts off power to 800 thousand customers...
2,30,0.266667,0.093023,0.222222,0.222222,PG&e shuts off power to 800 thousand customers...
3,35,0.360000,0.125000,0.240000,0.240000,PG&e shuts off power to 800 thousand customers...
4,40,0.407407,0.230769,0.333333,0.333333,PG&e shuts off power to 800 thousand customers...
5,45,0.444444,0.269231,0.370370,0.370370,PG&e shuts off power to 800 thousand customers...
6,50,0.444444,0.269231,0.370370,0.370370,PG&e shuts off power to 800 thousand customers...
7,55,0.444444,0.269231,0.370370,0.370370,PG&e shuts off power to 800 thousand customers...
8,60,0.444444,0.269231,0.370370,0.370370,PG&e shuts off power to 800 thousand customers...



Sentence for max_new_tokens = 20
'PG&e shuts off power to 800 thousand customers . the shutoffs are scheduled'

Sentence for max_new_tokens = 25
('PG&e shuts off power to 800 thousand customers . the shutoffs are scheduled '
 'to last through at least')

Sentence for max_new_tokens = 30
('PG&e shuts off power to 800 thousand customers . the shutoffs are scheduled '
 'to last through at least midday tomorrow .')

Sentence for max_new_tokens = 35
('PG&e shuts off power to 800 thousand customers . the shutoffs are scheduled '
 'to last through at least midday tomorrow . the aim is to reduce')

Sentence for max_new_tokens = 40
('PG&e shuts off power to 800 thousand customers . the shutoffs are scheduled '
 'to last through at least midday tomorrow . the aim is to reduce the risk of '
 'wildfire')

Sentence for max_new_tokens = 45
('PG&e shuts off power to 800 thousand customers . the shutoffs are scheduled '
 'to last through at least midday tomorrow . the aim is to reduce the risk of '
 

Testing num_beams from 2 to 10:
- With early_stopping = True

In [ ]:
# Updating hyperparameter to change
changed_hp = 'num_beams'
range_start = 2
range_stop = 10
step = 1

# Getting the df
df_2 = loop_through_hp(t5model, t5tokenizer, inputs["input_ids"],
  SHORT_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=40,
  num_beams=1, early_stopping=True, no_repeat_ngram_size=0, do_sample=False,
  top_k=50, temperature=1, top_p=1)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_2)

100%|██████████| 8/8 [02:40<00:00, 20.06s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,2,0.407407,0.230769,0.333333,0.333333,PG&e shuts off power to 800 thousand customers...
1,3,0.407407,0.230769,0.333333,0.333333,PG&e shuts off power to 800 thousand customers...
2,4,0.407407,0.230769,0.333333,0.333333,PG&e shuts off power to 800 thousand customers...
3,5,0.355556,0.232558,0.311111,0.311111,the shutoffs are scheduled to last through at ...
4,6,0.355556,0.232558,0.311111,0.311111,the shutoffs are scheduled to last through at ...
5,7,0.367347,0.212766,0.285714,0.285714,PG&e scheduled the blackouts in response to fo...
6,8,0.375000,0.217391,0.291667,0.291667,PG&e says it scheduled the blackouts in respon...
7,9,0.375000,0.217391,0.291667,0.291667,PG&e says it scheduled the blackouts in respon...



Sentence for num_beams = 2
('PG&e shuts off power to 800 thousand customers . the shutoffs are scheduled '
 'to last through at least midday tomorrow . the aim is to reduce the risk of '
 'wildfire')

Sentence for num_beams = 3
('PG&e shuts off power to 800 thousand customers . the shutoffs are scheduled '
 'to last through at least midday tomorrow . the aim is to reduce the risk of '
 'wildfire')

Sentence for num_beams = 4
('PG&e shuts off power to 800 thousand customers . the shutoffs are scheduled '
 'to last through at least midday tomorrow . the aim is to reduce the risk of '
 'wildfire')

Sentence for num_beams = 5
('the shutoffs are scheduled to last through at least midday tomorrow . the '
 'aim is to reduce the risk of wildfires .')

Sentence for num_beams = 6
('the shutoffs are scheduled to last through at least midday tomorrow . the '
 'aim is to reduce the risk of wildfires .')

Sentence for num_beams = 7
('PG&e scheduled the blackouts in response to forecasts for high wi

Testing no_repeat_ngram_size from 2 to 3:
- With the num_beams = 3 (from above)
- And early_stopping = True

In [ ]:
# Updating hyperparameter to change
changed_hp = 'no_repeat_ngram_size'
range_start = 2
range_stop = 4
step = 1

# Getting the df
df_3 = loop_through_hp(t5model, t5tokenizer, inputs["input_ids"],
  SHORT_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=40,
  num_beams=3, early_stopping=True, no_repeat_ngram_size=0, do_sample=False,
  top_k=50, temperature=1, top_p=1)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_3)

100%|██████████| 2/2 [00:41<00:00, 20.78s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,2,0.490566,0.313725,0.415094,0.415094,PG&e shuts off power to 800 thousand customers...
1,3,0.520000,0.333333,0.440000,0.440000,PG&e shuts off power to 800 thousand customers...



Sentence for no_repeat_ngram_size = 2
('PG&e shuts off power to 800 thousand customers in response to forecasts for '
 'high winds . the aim is to reduce the risk of wildfires, according to pg&')

Sentence for no_repeat_ngram_size = 3
('PG&e shuts off power to 800 thousand customers in response to forecasts for '
 'high winds . the aim is to reduce the risk of wildfires .')


Testing sampling as True:
- With top_k = 0

In [ ]:
# Updating hyperparameter to change
changed_hp = 'top_k'
range_start = 0
range_stop = 1
step = 1

# Getting the df
df_4 = loop_through_hp(t5model, t5tokenizer, inputs["input_ids"],
  SHORT_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=40,
  num_beams=1, early_stopping=False, no_repeat_ngram_size=0, do_sample=True,
  top_k=50, temperature=1, top_p=1)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_4)

100%|██████████| 1/1 [00:12<00:00, 12.40s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,0,0.192308,0.0,0.115385,0.115385,"a total of 800,000 customers will be affected ..."



Sentence for top_k = 0
('a total of 800,000 customers will be affected . the shutoffs are scheduled '
 'to stretch through then travel to this afternoon . fifty thousand homes '
 'could be affected .')


Testing temperature from 0.2 to 2
- With top_k = 0
- And do_sample = True

In [ ]:
# Updating hyperparameter to change
changed_hp = 'temperature'
range_start = 2
range_stop = 21
step = 2

# Getting the df
df_5 = loop_through_hp(t5model, t5tokenizer, inputs["input_ids"],
  SHORT_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=40,
  num_beams=1, early_stopping=False, no_repeat_ngram_size=0, do_sample=True,
  top_k=0, temperature=1, top_p=1)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_5)

100%|██████████| 10/10 [02:19<00:00, 13.94s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,2,0.407407,0.230769,0.333333,0.333333,PG&e shuts off power to 800 thousand customers...
1,4,0.346154,0.200000,0.269231,0.269231,PG&e scheduled the blackouts in response to fo...
2,6,0.163265,0.000000,0.122449,0.122449,the shutoffs are scheduled to last through at ...
3,8,0.301887,0.117647,0.226415,0.226415,PG&e shuts down 800 thousand customers to redu...
4,10,0.188679,0.000000,0.113208,0.113208,some 800k stutters were scheduled to be affect...
5,12,0.185185,0.000000,0.111111,0.111111,some 800 million customers due to be affected ...
6,14,0.145455,0.037736,0.109091,0.109091,shutouts included threat of dangerous wildfire...
7,16,0.039216,0.000000,0.039216,0.039216,main reasons clover nordberg chaos after oil s...
8,18,0.070175,0.000000,0.070175,0.070175,general updating: power component undidicated ...
9,20,0.000000,0.000000,0.000000,0.000000,enable danneau valleydruck tubes war on 200 on...



Sentence for temperature = 2
('PG&e shuts off power to 800 thousand customers . the shutoffs are scheduled '
 'to last through at least midday tomorrow . the aim is to reduce the risk of '
 'wildfire')

Sentence for temperature = 4
('PG&e scheduled the blackouts in response to forecasts for high winds . the '
 'aim is to reduce the risk of wildfires . the shutoffs are the fourth round')

Sentence for temperature = 6
('the shutoffs are scheduled to last through at least midday tomorrow . it is '
 'the fourth round of mass blackouts since october . PG&E blames')

Sentence for temperature = 8
('PG&e shuts down 800 thousand customers to reduce risk of wildfires . company '
 'says it is responding to forecasts for high winds amid dry conditions . '
 'shutoffs are third')

Sentence for temperature = 10
('some 800k stutters were scheduled to be affected for the next 4 days . the '
 'shutdowns are expected to last through at least midday tomorrow . suspending '
 'power will reduce')

Sentence

Testing top_k from 5 to 50:
- With do_sample = True

In [ ]:
# Updating hyperparameter to change
changed_hp = 'top_k'
range_start = 5
range_stop = 51
step = 5

# Getting the df
df_6 = loop_through_hp(t5model, t5tokenizer, inputs["input_ids"],
  SHORT_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=40,
  num_beams=1, early_stopping=False, no_repeat_ngram_size=0, do_sample=True,
  top_k=50, temperature=1, top_p=1)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_6)

100%|██████████| 10/10 [02:23<00:00, 14.37s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,5,0.240000,0.041667,0.200000,0.200000,PG&e shuts off power for nearly 800 thousand c...
1,10,0.444444,0.115385,0.296296,0.296296,PG&e shuts off customers from its power plants...
2,15,0.240000,0.041667,0.200000,0.200000,blackouts expected to last through at-midday t...
3,20,0.153846,0.000000,0.115385,0.115385,PG&E shuts in response to high winds amid dry ...
4,25,0.264151,0.078431,0.226415,0.226415,blackouts were scheduled to last through at le...
5,30,0.196078,0.000000,0.117647,0.117647,"PG&E shuts down nearly 800,000 customers due t..."
6,35,0.047619,0.000000,0.047619,0.047619,storms and dry conditions have left storm-land...
7,40,0.235294,0.000000,0.156863,0.156863,a record breaking drought threatens to increas...
8,45,0.120000,0.000000,0.080000,0.080000,sen. lamar campbell: the power blackouts are p...
9,50,0.156863,0.000000,0.117647,0.117647,storms threaten 800 thousand of PG&E customers...



Sentence for top_k = 5
('PG&e shuts off power for nearly 800 thousand customers . it is the fourth '
 'blackout of the public safety shutoff since october . a drought-related')

Sentence for top_k = 10
('PG&e shuts off customers from its power plants amid dry conditions due to '
 "heavy rainfall . the utility's plan is to cut the risk of wildfires . it's")

Sentence for top_k = 15
('blackouts expected to last through at-midday tomorrow . PG&e planned '
 'blackouts in response to forecasts for high wind . aim was to reduce '
 'wildfire risk')

Sentence for top_k = 20
('PG&E shuts in response to high winds amid dry conditions . shutoffs would be '
 'fourth round of mass blackouts imposed by utility since late last year . '
 'governor,')

Sentence for top_k = 25
('blackouts were scheduled to last through at least midday . pg&e is scheduled '
 'to continue its public safety shutoffs . the purpose of the action is to '
 'reduce the')

Sentence for top_k = 30
('PG&E shuts down nearly 800,00

Testing top_p from 0.1 to 0.9:
- With do_sample = True
- And top_k = 0

In [ ]:
# Updating hyperparameter to change
changed_hp = 'top_p'
range_start = 10
range_stop = 100
step = 10

# Getting the df
df_7 = loop_through_hp(t5model, t5tokenizer, inputs["input_ids"],
  SHORT_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=40,
  num_beams=1, early_stopping=False, no_repeat_ngram_size=0, do_sample=True,
  top_k=0, temperature=1, top_p=1)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_7)

100%|██████████| 9/9 [02:03<00:00, 13.77s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,10,0.407407,0.230769,0.333333,0.333333,PG&e shuts off power to 800 thousand customers...
1,20,0.407407,0.230769,0.333333,0.333333,PG&e shuts off power to 800 thousand customers...
2,30,0.407407,0.230769,0.333333,0.333333,PG&e shuts off power to 800 thousand customers...
3,40,0.407407,0.230769,0.333333,0.333333,PG&e shuts off power to 800 thousand customers...
4,50,0.296296,0.115385,0.259259,0.259259,PG&e shuts off power to 800 thousand customers...
5,60,0.339623,0.196078,0.301887,0.301887,PG&e shuts down power for almost 800 thousand ...
6,70,0.285714,0.037037,0.178571,0.178571,PG&e shuts down 800 thousand customers in resp...
7,80,0.226415,0.078431,0.226415,0.226415,utility shuts off power to 800k customers as d...
8,90,0.230769,0.000000,0.192308,0.192308,the grid is struggling to protect customers fr...



Sentence for top_p = 10
('PG&e shuts off power to 800 thousand customers . the shutoffs are scheduled '
 'to last through at least midday tomorrow . the aim is to reduce the risk of '
 'wildfire')

Sentence for top_p = 20
('PG&e shuts off power to 800 thousand customers . the shutoffs are scheduled '
 'to last through at least midday tomorrow . the aim is to reduce the risk of '
 'wildfire')

Sentence for top_p = 30
('PG&e shuts off power to 800 thousand customers . the shutoffs are scheduled '
 'to last through at least midday tomorrow . the aim is to reduce the risk of '
 'wildfire')

Sentence for top_p = 40
('PG&e shuts off power to 800 thousand customers . the shutoffs are scheduled '
 'to last through at least midday tomorrow . the aim is to reduce the risk of '
 'wildfire')

Sentence for top_p = 50
('PG&e shuts off power to 800 thousand customers in response to forecasts for '
 'high winds . the shutoffs are expected to last through at least midday '
 'tomorrow . the aim')

Sent

Trying best sampling values:
- With do_sample = True
- And top_p = 0.3
- And top_k = 10
- And temperature = 0.2

In [ ]:
# Updating hyperparameter to change
changed_hp = 'top_k'
range_start = 10
range_stop = 11
step = 1

# Getting the df
df_8 = loop_through_hp(t5model, t5tokenizer, inputs["input_ids"],
  SHORT_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=40,
  num_beams=1, early_stopping=False, no_repeat_ngram_size=0, do_sample=True,
  top_k=10, temperature=0.2, top_p=0.3)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_8)

100%|██████████| 1/1 [00:13<00:00, 13.17s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,10,0.407407,0.230769,0.333333,0.333333,PG&e shuts off power to 800 thousand customers...



Sentence for top_k = 10
('PG&e shuts off power to 800 thousand customers . the shutoffs are scheduled '
 'to last through at least midday tomorrow . the aim is to reduce the risk of '
 'wildfire')


Testing min_lengths from 10 to 60:
- With do_sample = True
- And top_p = 0.3
- And top_k = 10
- And temperature = 0.2

In [ ]:
# Updating hyperparameter to change
changed_hp = 'min_length'
range_start = 10
range_stop = 61
step = 10

# Getting the df
df_9 = loop_through_hp(t5model, t5tokenizer, inputs["input_ids"],
  SHORT_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=100,
  num_beams=1, early_stopping=False, no_repeat_ngram_size=0, do_sample=True,
  top_k=10, temperature=0.2, top_p=0.3)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_9)

100%|██████████| 6/6 [01:46<00:00, 17.73s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,10,0.444444,0.269231,0.370370,0.370370,PG&e shuts off power to 800 thousand customers...
1,20,0.444444,0.269231,0.370370,0.370370,PG&e shuts off power to 800 thousand customers...
2,30,0.444444,0.269231,0.370370,0.370370,PG&e shuts off power to 800 thousand customers...
3,40,0.444444,0.269231,0.370370,0.370370,PG&e shuts off power to 800 thousand customers...
4,50,0.375000,0.225806,0.312500,0.312500,PG&e shuts off power to 800 thousand customers...
5,60,0.342105,0.189189,0.263158,0.263158,PG&e shuts off power to 800 thousand customers...



Sentence for min_length = 10
('PG&e shuts off power to 800 thousand customers . the shutoffs are scheduled '
 'to last through at least midday tomorrow . the aim is to reduce the risk of '
 'wildfires .')

Sentence for min_length = 20
('PG&e shuts off power to 800 thousand customers . the shutoffs are scheduled '
 'to last through at least midday tomorrow . the aim is to reduce the risk of '
 'wildfires .')

Sentence for min_length = 30
('PG&e shuts off power to 800 thousand customers . the shutoffs are scheduled '
 'to last through at least midday tomorrow . the aim is to reduce the risk of '
 'wildfires .')

Sentence for min_length = 40
('PG&e shuts off power to 800 thousand customers . the shutoffs are scheduled '
 'to last through at least midday tomorrow . the aim is to reduce the risk of '
 'wildfires .')

Sentence for min_length = 50
('PG&e shuts off power to 800 thousand customers . the shutoffs are scheduled '
 'to last through at least midday tomorrow . the aim is to reduce 

Confirming again that beam search is still more effective:
- With early_stopping = True
- And num_beams = 3
- And no_repeat_ngram_size = 3

In [ ]:
# Updating hyperparameter to change
changed_hp = 'no_repeat_ngram_size'
range_start = 3
range_stop = 4
step = 1

# Getting the df
df_10 = loop_through_hp(t5model, t5tokenizer, inputs["input_ids"],
  SHORT_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=40,
  num_beams=3, early_stopping=True, no_repeat_ngram_size=3, do_sample=False,
  top_k=50, temperature=1, top_p=1)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_10)

100%|██████████| 1/1 [00:18<00:00, 18.79s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,3,0.52,0.333333,0.44,0.44,PG&e shuts off power to 800 thousand customers...



Sentence for no_repeat_ngram_size = 3
('PG&e shuts off power to 800 thousand customers in response to forecasts for '
 'high winds . the aim is to reduce the risk of wildfires .')


**QUESTION:**

1.1 What num_beams value gives you the most readable output that meets the score criteria?
- 3

1.2 Which no_repeat_ngram_size gives the most readable output that meets the score criteria?
- 3

1.3 What min_length value gives you the most readable output that meets the score criteria?
- 10

1.4 Which max_new_tokens value gives the most readable output that meets the score criteria?
- 50

1.5 What is the ROUGE-L score associated with your most readable candidate?
- 0.44000

In [19]:
#In order to not consume all of the memory available in Colab we'll free up the memory we're using for these large language models
del t5model
del t5tokenizer


## 2. Pegasus for Headline Summarization

Pegasus is an encoder decoder architecture that has been explicitly pre-trained as an abstractive summarizer.  You can read more about it [here](https://huggingface.co/docs/transformers/model_doc/pegasus) and [here](https://arxiv.org/pdf/1912.08777.pdf).

We'll first use the `google/pegasus-xsum` checkpoint.  It is trained on a [summarization task](https://aclanthology.org/D18-1206.pdf) that reads a news article and then [emits a one line summary](https://huggingface.co/datasets/xsum).  This doesn't mean that it is limited in its output length.  It does mean that it works well with news article type inputs and tends toward shorter outputs.

In [20]:
from transformers import PegasusTokenizer, TFPegasusForConditionalGeneration

pmodel = TFPegasusForConditionalGeneration.from_pretrained("google/pegasus-xsum")
ptokenizer = PegasusTokenizer.from_pretrained("google/pegasus-xsum")

config.json: 0.00B [00:00, ?B/s]

tf_model.h5:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

All model checkpoint layers were used when initializing TFPegasusForConditionalGeneration.

Some layers of TFPegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['final_logits_bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/259 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/87.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [16]:
pmodel.summary()

Model: "tf_pegasus_for_conditional_generation"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 model (TFPegasusMainLayer)  multiple                  569748480 
                                                                 
 final_logits_bias (BiasLay  multiple                  96103     
 er)                                                             
                                                                 
Total params: 569844583 (2.12 GB)
Trainable params: 569748480 (2.12 GB)
Non-trainable params: 96103 (375.40 KB)
_________________________________________________________________


Let's see what kinds of default parameters are configured in to this checkpoint.

In [21]:
config = AutoConfig.from_pretrained("google/pegasus-xsum")

config

PegasusConfig {
  "activation_dropout": 0.1,
  "activation_function": "relu",
  "add_bias_logits": false,
  "add_final_layer_norm": true,
  "architectures": [
    "PegasusForConditionalGeneration"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": 0,
  "classif_dropout": 0.0,
  "classifier_dropout": 0.0,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 16,
  "decoder_start_token_id": 0,
  "do_blenderbot_90_layernorm": false,
  "dropout": 0.1,
  "encoder_attention_heads": 16,
  "encoder_ffn_dim": 4096,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 16,
  "eos_token_id": 1,
  "extra_pos_embeddings": 0,
  "force_bos_token_to_be_generated": false,
  "forced_eos_token_id": 1,
  "gradient_checkpointing": false,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "init_std": 0.02,
  "is_encoder_decoder": true,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2
  },
  

Generate the inputs using the pegasus tokenizer for this checkpoint.

In [22]:
inputs = ptokenizer(ARTICLE_TO_SUMMARIZE, max_length=1024, truncation=True, return_tensors="tf")

In [23]:
inputs

{'input_ids': <tf.Tensor: shape=(1, 209), dtype=int32, numpy=
array([[16502,  6194,  4927,   527,   127,  2798,   112,   129,  2790,
          141,   109, 87338,   116,   162,   127,  1214,   112,   289,
          224,   134,   583, 26568,  3469,   107, 14887,   759,  1005,
         3163,   126,  2798,   109, 25690,   116,   115,  1407,   112,
        13378,   118,   281,  7213, 10754,  1514,  1047,   107,   139,
         2560,   117,   112,  1329,   109,   887,   113, 39471,   107,
          240,  3755,  5259,   259,  6138,  1398,   108,   114,  1451,
          113, 14887,   759,  1005,  7319,   108,  1168,   224,   122,
          372,   481,  1008,   484, 87338,   108,   126,   192,   129,
          109,  2868,  1344,   113,  2977, 25690,   116, 10253,   141,
          109,  3826,   381,  5177,   107,  8283,   173,   181,   624,
        31894,   527,   195,   518,   115,   109,  1700,   107,   139,
          909,  4660,   113, 50340, 87338,   116,   133,  4188,  4565,
         7881, 

Let's get some output using just the default values and see what we're working with.

In [20]:
# Generate Summary
summary_ids = pmodel.generate(inputs["input_ids"]
)
pprint(ptokenizer.batch_decode(summary_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0], compact=True)

("California's largest utility has announced plans to cut power to hundreds of "
 'thousands of customers in a bid to reduce the risk of wildfires.')


Let's experiment with the same set of hyperparameters for the Pegasus system.  It is designed for abstractive summarization. Remember that the checkpoint we are using was trained on data that generates a one line summary for the input article.

*Your readable output should consist of at least one complete sentence but does not have to end with a period and you must also have a ROUGE-1 score above 0.30 and ROUGE-L score equal to or above 0.25 when compared with the short reference.*

You can use the two cells below to experiment with hyperparameters and generating and scoring your outputs in order to answer questions 2.1 - 2.5 in your answers file.

**QUESTION:**

2.1 What num_beams value gives you the most readable output that meets the score criteria?
- 7

2.2 Which no_repeat_ngram_size gives the most readable output that meets the score criteria?
- 3

2.3 What min_length value gives you the most readable output that meets the score criteria?
- 10

2.4 Which max_new_tokens value gives the most readable output that meets the score criteria?
- 40

2.5 What is the ROUGE-L score associated with your most readable candidate?
- 0.84000

In [25]:
SHORT_REFERENCE

"California's largest utility is set to turn off power to hundreds of thousands of customers in an effort to reduce the risk of wildfires. "

In [21]:
# Generate Summary
summary_ids = pmodel.generate(inputs["input_ids"],
### YOUR CODE HERE
                              max_new_tokens=40
### END YOUR CODE
)
candidate = ptokenizer.batch_decode(summary_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
pprint(candidate[0], compact=True)

("California's largest utility has announced plans to cut power to hundreds of "
 'thousands of customers in a bid to reduce the risk of wildfires.')


In [22]:
rouge = evaluate.load('rouge')
predictions = candidate
references = [SHORT_REFERENCE]
results = rouge.compute(predictions=predictions,
                        references=references)
print(results)

{'rouge1': np.float64(0.76), 'rouge2': np.float64(0.625), 'rougeL': np.float64(0.76), 'rougeLsum': np.float64(0.76)}


**Once again, I will use my functions above to test all the hyperparameters individually.**

Testing max_new_tokens from 20 to 60:

In [26]:
# Updating hyperparameter to change
changed_hp = 'max_new_tokens'
range_start = 20
range_stop = 61
step = 5

# Getting the df
df_1 = loop_through_hp(pmodel, ptokenizer, inputs["input_ids"],
  SHORT_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=40,
  num_beams=1, early_stopping=False, no_repeat_ngram_size=0, do_sample=False,
  top_k=50, temperature=1, top_p=1)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_1)

100%|██████████| 9/9 [04:45<00:00, 31.76s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,20,0.511628,0.195122,0.465116,0.465116,California's largest utility is preparing to i...
1,25,0.510638,0.177778,0.468085,0.468085,California's largest utility is preparing to i...
2,30,0.510638,0.177778,0.468085,0.468085,California's largest utility is preparing to i...
3,35,0.510638,0.177778,0.468085,0.468085,California's largest utility is preparing to i...
4,40,0.510638,0.177778,0.468085,0.468085,California's largest utility is preparing to i...
5,45,0.510638,0.177778,0.468085,0.468085,California's largest utility is preparing to i...
6,50,0.510638,0.177778,0.468085,0.468085,California's largest utility is preparing to i...
7,55,0.510638,0.177778,0.468085,0.468085,California's largest utility is preparing to i...
8,60,0.510638,0.177778,0.468085,0.468085,California's largest utility is preparing to i...



Sentence for max_new_tokens = 20
("California's largest utility is preparing to impose a series of power cuts "
 'to customers amid fears of')

Sentence for max_new_tokens = 25
("California's largest utility is preparing to impose a series of power cuts "
 'to customers amid fears of high winds and wildfires.')

Sentence for max_new_tokens = 30
("California's largest utility is preparing to impose a series of power cuts "
 'to customers amid fears of high winds and wildfires.')

Sentence for max_new_tokens = 35
("California's largest utility is preparing to impose a series of power cuts "
 'to customers amid fears of high winds and wildfires.')

Sentence for max_new_tokens = 40
("California's largest utility is preparing to impose a series of power cuts "
 'to customers amid fears of high winds and wildfires.')

Sentence for max_new_tokens = 45
("California's largest utility is preparing to impose a series of power cuts "
 'to customers amid fears of high winds and wildfires.')

Sente

Testing num_beams from 2 to 10:
- With early_stopping = True

In [27]:
# Updating hyperparameter to change
changed_hp = 'num_beams'
range_start = 2
range_stop = 10
step = 1

# Getting the df
df_2 = loop_through_hp(pmodel, ptokenizer, inputs["input_ids"],
  SHORT_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=40,
  num_beams=1, early_stopping=True, no_repeat_ngram_size=0, do_sample=False,
  top_k=50, temperature=1, top_p=1)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_2)

100%|██████████| 8/8 [06:16<00:00, 47.04s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,2,0.478261,0.318182,0.434783,0.434783,California's largest utility is preparing for ...
1,3,0.720000,0.625000,0.720000,0.720000,California's largest utility has announced it ...
2,4,0.760000,0.625000,0.760000,0.760000,California's largest utility has announced pla...
3,5,0.760000,0.625000,0.760000,0.760000,California's largest utility has announced pla...
4,6,0.760000,0.625000,0.760000,0.760000,California's largest utility has announced pla...
5,7,0.840000,0.750000,0.840000,0.840000,California's largest utility has announced pla...
6,8,0.760000,0.625000,0.760000,0.760000,California's largest utility has announced pla...
7,9,0.760000,0.625000,0.760000,0.760000,California's largest utility has announced pla...



Sentence for num_beams = 2
("California's largest utility is preparing for a wave of power blackouts as a "
 'precautionary measure amid the risk of wildfires.')

Sentence for num_beams = 3
("California's largest utility has announced it will cut power to hundreds of "
 'thousands of customers in a bid to reduce the risk of wildfires.')

Sentence for num_beams = 4
("California's largest utility has announced plans to cut power to hundreds of "
 'thousands of customers in a bid to reduce the risk of wildfires.')

Sentence for num_beams = 5
("California's largest utility has announced plans to cut power to hundreds of "
 'thousands of customers in a bid to reduce the risk of wildfires.')

Sentence for num_beams = 6
("California's largest utility has announced plans to cut power to hundreds of "
 'thousands of customers in a bid to reduce the risk of wildfires.')

Sentence for num_beams = 7
("California's largest utility has announced plans to cut power to hundreds of "
 'thousands of cu

Testing no_repeat_ngram_size from 2 to 3:
- With the num_beams = 7 (from above)
- And early_stopping = True

In [31]:
# Updating hyperparameter to change
changed_hp = 'no_repeat_ngram_size'
range_start = 2
range_stop = 4
step = 1

# Getting the df
df_3 = loop_through_hp(pmodel, ptokenizer, inputs["input_ids"],
  SHORT_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=40,
  num_beams=7, early_stopping=True, no_repeat_ngram_size=0, do_sample=False,
  top_k=50, temperature=1, top_p=1)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_3)

100%|██████████| 2/2 [02:02<00:00, 61.23s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,2,0.84,0.75,0.84,0.84,California's largest utility has announced pla...
1,3,0.84,0.75,0.84,0.84,California's largest utility has announced pla...



Sentence for no_repeat_ngram_size = 2
("California's largest utility has announced plans to cut power to hundreds of "
 'thousands of customers in an effort to reduce the risk of wildfires.')

Sentence for no_repeat_ngram_size = 3
("California's largest utility has announced plans to cut power to hundreds of "
 'thousands of customers in an effort to reduce the risk of wildfires.')


Testing sampling as True:
- With top_k = 0

In [29]:
# Updating hyperparameter to change
changed_hp = 'top_k'
range_start = 0
range_stop = 1
step = 1

# Getting the df
df_4 = loop_through_hp(pmodel, ptokenizer, inputs["input_ids"],
  SHORT_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=40,
  num_beams=1, early_stopping=False, no_repeat_ngram_size=0, do_sample=True,
  top_k=50, temperature=1, top_p=1)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_4)

100%|██████████| 1/1 [00:20<00:00, 20.11s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,0,0.052632,0.0,0.052632,0.052632,line PSDS-07-17926-1 insulate cardboard cocoon...



Sentence for top_k = 0
'line PSDS-07-17926-1 insulate cardboard cocoons emailed to Wills, M-23)'


Testing temperature from 0.2 to 2
- With top_k = 0
- And do_sample = True

In [32]:
# Updating hyperparameter to change
changed_hp = 'temperature'
range_start = 2
range_stop = 21
step = 2

# Getting the df
df_5 = loop_through_hp(pmodel, ptokenizer, inputs["input_ids"],
  SHORT_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=40,
  num_beams=1, early_stopping=False, no_repeat_ngram_size=0, do_sample=True,
  top_k=0, temperature=1, top_p=1)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_5)

100%|██████████| 10/10 [05:28<00:00, 32.89s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,2,0.538462,0.400000,0.538462,0.538462,California's largest utility is cutting power ...
1,4,0.431373,0.163265,0.313725,0.313725,Power will be turned off to millions of custom...
2,6,0.553191,0.355556,0.553191,0.553191,California's largest utility has announced it ...
3,8,0.177778,0.000000,0.133333,0.133333,Utilities in California are preparing for heav...
4,10,0.382979,0.133333,0.340426,0.340426,Utilities in California had to shut off power ...
5,12,0.000000,0.000000,0.000000,0.000000,Or hoped anyscore progresses with competitionO...
6,14,0.037736,0.000000,0.037736,0.037736,Supervisors in Athletics Linkage DEPARTMENT fr...
7,16,0.039216,0.000000,0.039216,0.039216,vacationplan crdf transactincome AirPlay datte...
8,18,0.038462,0.000000,0.038462,0.038462,DharmaBreakfast covers MSNBC-SC polynomialCOME...
9,20,0.000000,0.000000,0.000000,0.000000,cabinet contention Synth galettelifeHDRBTC kic...



Sentence for temperature = 2
("California's largest utility is cutting power to hundreds of thousands of "
 'customers as it prepares for a new round of high winds that could fuel '
 'wildfires.')

Sentence for temperature = 4
('Power will be turned off to millions of customers across California for the '
 'next few days amid warnings of high winds and the risk of wildfires.')

Sentence for temperature = 6
("California's largest utility has announced it is cutting power to millions "
 'of customers amid growing concerns over the risk of wildfires.')

Sentence for temperature = 8
('Utilities in California are preparing for heavy, prolonged power cuts last '
 'night as a result of drought and high winds.')

Sentence for temperature = 10
('Utilities in California had to shut off power to hundreds META,000 customer '
 'accounts amid record Cambriacommunity drought sequel to recent wildfires.')

Sentence for temperature = 12
('Or hoped anyscore progresses with competitionOKENIntelligent sa

Testing top_k from 5 to 50:
- With do_sample = True

In [33]:
# Updating hyperparameter to change
changed_hp = 'top_k'
range_start = 5
range_stop = 51
step = 5

# Getting the df
df_6 = loop_through_hp(pmodel, ptokenizer, inputs["input_ids"],
  SHORT_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=40,
  num_beams=1, early_stopping=False, no_repeat_ngram_size=0, do_sample=True,
  top_k=50, temperature=1, top_p=1)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_6)

100%|██████████| 10/10 [05:02<00:00, 30.23s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,5,0.488889,0.186047,0.355556,0.355556,Power is to be cut for thousands of customers ...
1,10,0.545455,0.377358,0.545455,0.545455,California's largest utility is scheduled to s...
2,15,0.339623,0.000000,0.188679,0.188679,California customers of two major power compan...
3,20,0.444444,0.230769,0.333333,0.333333,California's two biggest utilities have declar...
4,25,0.274510,0.040816,0.156863,0.156863,California utilities have declared states of e...
5,30,0.269231,0.040000,0.192308,0.192308,A series of power outages are likely in Califo...
6,35,0.240000,0.041667,0.120000,0.120000,Thousands of California residents will be plun...
7,40,0.250000,0.074074,0.178571,0.178571,One of California's utilities has told its 1.2...
8,45,0.291667,0.130435,0.291667,0.291667,California's power utilities are facing a mass...
9,50,0.235294,0.040816,0.196078,0.196078,Several hundred thousand Northern California h...



Sentence for top_k = 5
('Power is to be cut for thousands of customers in California amid fears of '
 'wildfires and record levels of dryness.')

Sentence for top_k = 10
("California's largest utility is scheduled to start cutting power to hundreds "
 'of thousands of customers last night amid the latest wave of criticism of '
 'its safety record during drought.')

Sentence for top_k = 15
('California customers of two major power companies are in danger of being '
 'left without electricity for up to a week as drought and wildfires spread '
 'across the state.')

Sentence for top_k = 20
("California's two biggest utilities have declared power cuts in response to "
 'drought, high winds and the risk of wildfires to reduce the scope of '
 'blackouts for their clients.')

Sentence for top_k = 25
('California utilities have declared states of emergency for at least a week '
 'as a series of wildfires threaten to spread and lead to major power '
 'blackouts.')

Sentence for top_k = 30
('A 

Testing top_p from 0.1 to 0.9:
- With do_sample = True
- And top_k = 0

In [34]:
# Updating hyperparameter to change
changed_hp = 'top_p'
range_start = 10
range_stop = 100
step = 10

# Getting the df
df_7 = loop_through_hp(pmodel, ptokenizer, inputs["input_ids"],
  SHORT_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=40,
  num_beams=1, early_stopping=False, no_repeat_ngram_size=0, do_sample=True,
  top_k=0, temperature=1, top_p=1)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_7)

100%|██████████| 9/9 [04:20<00:00, 28.99s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,10,0.510638,0.177778,0.468085,0.468085,California's largest utility is preparing to i...
1,20,0.549020,0.367347,0.509804,0.509804,California's largest utility is set to impose ...
2,30,0.489796,0.170213,0.448980,0.448980,California's largest utility is expected to im...
3,40,0.500000,0.260870,0.500000,0.500000,California's largest utility is preparing to i...
4,50,0.538462,0.320000,0.461538,0.461538,The state's largest utility is scheduled to be...
5,60,0.375000,0.130435,0.291667,0.291667,The biggest US power company is preparing to i...
6,70,0.333333,0.130435,0.208333,0.208333,Thousands of customers in California are facin...
7,80,0.391304,0.227273,0.347826,0.347826,Hundreds of thousands of California households...
8,90,0.448980,0.297872,0.448980,0.448980,Utilities have been ordered by California regu...



Sentence for top_p = 10
("California's largest utility is preparing to impose a series of power cuts "
 'to customers amid fears of high winds and wildfires.')

Sentence for top_p = 20
("California's largest utility is set to impose a series of massive power cuts "
 'as a result of the ongoing drought and the risk of wildfires.')

Sentence for top_p = 30
("California's largest utility is expected to impose a wave of power cuts to "
 'customers last night amid fears of high winds and wildfires.')

Sentence for top_p = 40
("California's largest utility is preparing to impose a series of large "
 'blackouts in response to high winds and a risk of wildfires.')

Sentence for top_p = 50
("The state's largest utility is scheduled to begin a massive power shut-off "
 'to reduce the risk of wildfires amid the worst drought in the US.')

Sentence for top_p = 60
('The biggest US power company is preparing to impose more blackouts in '
 'California amid a record-breaking drought and the risk of w

Trying best sampling values:
- With do_sample = True
- And top_p = 0.2
- And top_k = 10
- And temperature = 0.6

In [36]:
# Updating hyperparameter to change
changed_hp = 'top_k'
range_start = 10
range_stop = 11
step = 1

# Getting the df
df_8 = loop_through_hp(pmodel, ptokenizer, inputs["input_ids"],
  SHORT_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=40,
  num_beams=1, early_stopping=False, no_repeat_ngram_size=0, do_sample=True,
  top_k=10, temperature=0.6, top_p=0.2)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_8)

100%|██████████| 1/1 [00:27<00:00, 27.00s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,10,0.510638,0.177778,0.468085,0.468085,California's largest utility is preparing to i...



Sentence for top_k = 10
("California's largest utility is preparing to impose a series of power cuts "
 'to customers amid fears of high winds and wildfires.')


Testing min_lengths from 10 to 60:
- With do_sample = True
- And top_p = 0.2
- And top_k = 10
- And temperature = 0.6

In [37]:
# Updating hyperparameter to change
changed_hp = 'min_length'
range_start = 10
range_stop = 61
step = 10

# Getting the df
df_9 = loop_through_hp(pmodel, ptokenizer, inputs["input_ids"],
  SHORT_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=100,
  num_beams=1, early_stopping=False, no_repeat_ngram_size=0, do_sample=True,
  top_k=10, temperature=0.6, top_p=0.2)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_9)

100%|██████████| 6/6 [04:15<00:00, 42.66s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,10,0.510638,0.177778,0.468085,0.468085,California's largest utility is preparing to i...
1,20,0.510638,0.177778,0.468085,0.468085,California's largest utility is preparing to i...
2,30,0.459016,0.135593,0.360656,0.360656,California's largest utility is preparing to i...
3,40,0.459016,0.135593,0.360656,0.360656,California's largest utility is preparing to i...
4,50,0.373333,0.109589,0.320000,0.320000,California's largest utility is preparing to i...
5,60,0.373333,0.109589,0.320000,0.320000,California's largest utility is preparing to i...



Sentence for min_length = 10
("California's largest utility is preparing to impose a series of power cuts "
 'to customers amid fears of high winds and wildfires.')

Sentence for min_length = 20
("California's largest utility is preparing to impose a series of power cuts "
 'to customers amid fears of high winds and wildfires.')

Sentence for min_length = 30
("California's largest utility is preparing to impose a series of power cuts "
 'to customers amid fears of high winds and wildfires.., and is facing '
 'criticism from the governor for doing too much to prevent wildfires.')

Sentence for min_length = 40
("California's largest utility is preparing to impose a series of power cuts "
 'to customers amid fears of high winds and wildfires.., and is facing '
 'criticism from the governor for doing too much to prevent wildfires.')

Sentence for min_length = 50
("California's largest utility is preparing to impose a series of power cuts "
 'to customers amid fears of high winds and wildf

Confirming again that beam search is still more effective:
- With early_stopping = True
- And num_beams = 7
- And no_repeat_ngram_size = 3

In [35]:
# Updating hyperparameter to change
changed_hp = 'no_repeat_ngram_size'
range_start = 3
range_stop = 4
step = 1

# Getting the df
df_10 = loop_through_hp(pmodel, ptokenizer, inputs["input_ids"],
  SHORT_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=40,
  num_beams=7, early_stopping=True, no_repeat_ngram_size=3, do_sample=False,
  top_k=50, temperature=1, top_p=1)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_10)

100%|██████████| 1/1 [01:06<00:00, 66.05s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,3,0.84,0.75,0.84,0.84,California's largest utility has announced pla...



Sentence for no_repeat_ngram_size = 3
("California's largest utility has announced plans to cut power to hundreds of "
 'thousands of customers in an effort to reduce the risk of wildfires.')


Delete that Pegasus model and tokenizer so we can load the next one.

In [24]:
del pmodel
del ptokenizer

## 3. Pegasus for Longer Generation

Now let's try to produce a longer summary of our article.  In order to do that we are going to use a different fine-tuned checkpoint for Pegasus.  This checkpoint is fine-tuned on the [CNN/Daily Mail](https://huggingface.co/datasets/cnn_dailymail) set of news articles.  The references are on the order of several sentences long.

In [25]:
from transformers import PegasusTokenizer, TFPegasusForConditionalGeneration

cnnmodel = TFPegasusForConditionalGeneration.from_pretrained("google/pegasus-cnn_dailymail", from_pt=True)
cnntokenizer = PegasusTokenizer.from_pretrained("google/pegasus-cnn_dailymail", from_pt=True)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

All PyTorch model weights were used when initializing TFPegasusForConditionalGeneration.

Some weights or buffers of the TF 2.0 model TFPegasusForConditionalGeneration were not initialized from the PyTorch model and are newly initialized: ['model.encoder.embed_positions.weight', 'model.decoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

My notes on why we use from_pt=True:
- A path or url to a PyTorch state_dict save file (e.g, ./pt_model/pytorch_model.bin). In this case, from_pt should be set to True and a configuration object should be provided as config argument. This loading path is slower than converting the PyTorch model in a TensorFlow model using the provided conversion scripts and loading the TensorFlow model afterwards.

Let's see how this checkpoint is configured by default:

In [26]:
config = AutoConfig.from_pretrained("google/pegasus-cnn_dailymail")

config

PegasusConfig {
  "activation_dropout": 0.1,
  "activation_function": "relu",
  "add_bias_logits": false,
  "add_final_layer_norm": true,
  "architectures": [
    "PegasusForConditionalGeneration"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": 0,
  "classif_dropout": 0.0,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 16,
  "decoder_start_token_id": 0,
  "dropout": 0.1,
  "encoder_attention_heads": 16,
  "encoder_ffn_dim": 4096,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 16,
  "eos_token_id": 1,
  "extra_pos_embeddings": 1,
  "forced_eos_token_id": 1,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "init_std": 0.02,
  "is_encoder_decoder": true,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2
  },
  "length_penalty": 0.8,
  "max_length": 128,
  "max_position_embeddings": 1024,
  "min_length": 32,
  "model_type": "pegasus",
  "normalize_before":

Let's tokenize our input for this checkpoint.

In [27]:
cnninputs = cnntokenizer(ARTICLE_TO_SUMMARIZE, max_length=1024, truncation=True, return_tensors="tf")

Run the summarizer with the defaults and let's see what it looks like.

In [28]:
# Generate Summary
summary_ids = cnnmodel.generate(inputs["input_ids"]
)

pprint(cnntokenizer.batch_decode(summary_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0], compact=True)

('Nearly 800 thousand customers are scheduled to be affected by the shutoffs '
 'which are expected to last through at least midday tomorrow .<n>PG&E stated '
 'it scheduled the blackouts in response to forecasts for high winds amid dry '
 'conditions .<n>The aim is to reduce the risk of wildfires .')


Let's again experiment with the same set of hyperparameters (but possibly with different values) for the Pegasus system.  It is designed for abstractive summarization and this checkpoint is based on multi-line outputs.  We'll evaluate it against the long reference record.

*Your readable multi-line output must have a ROUGE-1 score above 0.25 and a ROUGE-L score above 0.15.*

You can use the two cells below to experiment with hyperparameters and generating and scoring your outputs in order to answer questions 3.1 - 3.5 in your answers file.

**QUESTION:**

3.1 What num_beams value gives you the most readable output that meets the score criteria?
- 4

3.2 Which no_repeat_ngram_size gives the most readable output that meets the score criteria?
- 3

3.3 What min_length value gives you the most readable output that meets the score criteria?
- 30

3.4 Which max_new_tokens value gives you the most readable output that meets the score criteria?
- 60

3.5 What is the ROUGE-L score associated with your most readable candidate?
- 0.35052

In [29]:
LONG_REFERENCE

'Many PG&E customers could be affected by public safety power shutoffs in response to forecasts for high winds and dry conditions. The record breaking drought exponentially increases the probability of large scale wildfires. Despite being criticized by Governor Newsom for being overly broad, company officials defend the cutoffs as a matter of public safety. '

In [30]:
# Generate Summary
summary_ids = cnnmodel.generate(cnninputs["input_ids"],
### YOUR CODE HERE
                                max_new_tokens=100
### END YOUR CODE
                             )
candidate = cnntokenizer.batch_decode(summary_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
pprint(candidate[0], compact=True)

('Nearly 800 thousand customers are scheduled to be affected by the shutoffs '
 'which are expected to last through at least midday tomorrow .<n>PG&E stated '
 'it scheduled the blackouts in response to forecasts for high winds amid dry '
 'conditions .<n>The aim is to reduce the risk of wildfires .')


In [31]:
rouge = evaluate.load('rouge')
predictions = candidate
references = [LONG_REFERENCE]
results = rouge.compute(predictions=predictions,
                        references=references)
print(results)

{'rouge1': np.float64(0.4000000000000001), 'rouge2': np.float64(0.19417475728155342), 'rougeL': np.float64(0.34285714285714286), 'rougeLsum': np.float64(0.34285714285714286)}


**Once again, I will use my functions above to test all the hyperparameters individually.**

Testing max_new_tokens from 40 to 80:

In [32]:
# Updating hyperparameter to change
changed_hp = 'max_new_tokens'
range_start = 40
range_stop = 81
step = 10

# Getting the df
df_1 = loop_through_hp(cnnmodel, cnntokenizer, cnninputs["input_ids"],
  LONG_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=60,
  num_beams=1, early_stopping=False, no_repeat_ngram_size=0, do_sample=False,
  top_k=50, temperature=1, top_p=1)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_1)

100%|██████████| 5/5 [04:17<00:00, 51.57s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,40,0.295455,0.069767,0.181818,0.181818,The aim of the shutoffs is to reduce the risk ...
1,50,0.306122,0.062500,0.163265,0.163265,The aim of the shutoffs is to reduce the risk ...
2,60,0.306122,0.062500,0.163265,0.163265,The aim of the shutoffs is to reduce the risk ...
3,70,0.306122,0.062500,0.163265,0.163265,The aim of the shutoffs is to reduce the risk ...
4,80,0.306122,0.062500,0.163265,0.163265,The aim of the shutoffs is to reduce the risk ...



Sentence for max_new_tokens = 40
('The aim of the shutoffs is to reduce the risk of wildfires .<n>The recent '
 'wave of precautionary shutoffs have drawn sharp criticism from Governor '
 'Gavin Newsom .<n>The record breaking drought has')

Sentence for max_new_tokens = 50
('The aim of the shutoffs is to reduce the risk of wildfires .<n>The recent '
 'wave of precautionary shutoffs have drawn sharp criticism from Governor '
 'Gavin Newsom .<n>The record breaking drought has made the current conditions '
 'even worse than in previous years')

Sentence for max_new_tokens = 60
('The aim of the shutoffs is to reduce the risk of wildfires .<n>The recent '
 'wave of precautionary shutoffs have drawn sharp criticism from Governor '
 'Gavin Newsom .<n>The record breaking drought has made the current conditions '
 'even worse than in previous years .')

Sentence for max_new_tokens = 70
('The aim of the shutoffs is to reduce the risk of wildfires .<n>The recent '
 'wave of precautionary shutoff

Testing num_beams from 2 to 10:
- With early_stopping = True

In [33]:
# Updating hyperparameter to change
changed_hp = 'num_beams'
range_start = 2
range_stop = 10
step = 1

# Getting the df
df_2 = loop_through_hp(cnnmodel, cnntokenizer, cnninputs["input_ids"],
  LONG_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=60,
  num_beams=1, early_stopping=True, no_repeat_ngram_size=0, do_sample=False,
  top_k=50, temperature=1, top_p=1)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_2)

100%|██████████| 8/8 [09:46<00:00, 73.33s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,2,0.387097,0.219780,0.301075,0.301075,The aim of the shutoffs is to reduce the risk ...
1,3,0.275000,0.076923,0.150000,0.150000,The aim is to reduce the risk of wildfires .<n...
2,4,0.378947,0.215054,0.294737,0.294737,Nearly 800 thousand customers are scheduled to...
3,5,0.378947,0.215054,0.294737,0.294737,Nearly 800 thousand customers are scheduled to...
4,6,0.378947,0.215054,0.294737,0.294737,Nearly 800 thousand customers are scheduled to...
5,7,0.378947,0.215054,0.294737,0.294737,Nearly 800 thousand customers are scheduled to...
6,8,0.378947,0.215054,0.294737,0.294737,Nearly 800 thousand customers are scheduled to...
7,9,0.378947,0.215054,0.294737,0.294737,Nearly 800 thousand customers are scheduled to...



Sentence for num_beams = 2
('The aim of the shutoffs is to reduce the risk of wildfires .<n>The record '
 'breaking drought has made the current conditions even worse than in previous '
 'years .<n>It exponentially increases the probability of large scale '
 'wildfires .')

Sentence for num_beams = 3
('The aim is to reduce the risk of wildfires .<n>The record breaking drought '
 'has made the current conditions even worse than in previous years .')

Sentence for num_beams = 4
('Nearly 800 thousand customers are scheduled to be affected by the shutoffs '
 'which are expected to last through at least midday tomorrow .<n>PG&E stated '
 'it scheduled the blackouts in response to forecasts for high winds amid dry '
 'conditions .')

Sentence for num_beams = 5
('Nearly 800 thousand customers are scheduled to be affected by the shutoffs '
 'which are expected to last through at least midday tomorrow .<n>PG&E stated '
 'it scheduled the blackouts in response to forecasts for high winds amid d

Testing no_repeat_ngram_size from 2 to 3:
- With the num_beams = 4 (anything 4 and after produced the same results from above)
- And early_stopping = True

In [38]:
# Updating hyperparameter to change
changed_hp = 'no_repeat_ngram_size'
range_start = 2
range_stop = 4
step = 1

# Getting the df
df_3 = loop_through_hp(cnnmodel, cnntokenizer, cnninputs["input_ids"],
  LONG_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=60,
  num_beams=4, early_stopping=True, no_repeat_ngram_size=0, do_sample=False,
  top_k=50, temperature=1, top_p=1)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_3)

100%|██████████| 2/2 [02:30<00:00, 75.28s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,2,0.378947,0.215054,0.294737,0.294737,Nearly 800 thousand customers are scheduled to...
1,3,0.378947,0.215054,0.294737,0.294737,Nearly 800 thousand customers are scheduled to...



Sentence for no_repeat_ngram_size = 2
('Nearly 800 thousand customers are scheduled to be affected by the shutoffs '
 'which are expected to last through at least midday tomorrow .<n>PG&E stated '
 'it scheduled the blackouts in response to forecasts for high winds amid dry '
 'conditions.')

Sentence for no_repeat_ngram_size = 3
('Nearly 800 thousand customers are scheduled to be affected by the shutoffs '
 'which are expected to last through at least midday tomorrow .<n>PG&E stated '
 'it scheduled the blackouts in response to forecasts for high winds amid dry '
 'conditions .')


Testing sampling as True:
- With top_k = 0

In [34]:
# Updating hyperparameter to change
changed_hp = 'top_k'
range_start = 0
range_stop = 1
step = 1

# Getting the df
df_4 = loop_through_hp(cnnmodel, cnntokenizer, cnninputs["input_ids"],
  LONG_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=60,
  num_beams=1, early_stopping=False, no_repeat_ngram_size=0, do_sample=True,
  top_k=50, temperature=1, top_p=1)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_4)

100%|██████████| 1/1 [00:30<00:00, 30.51s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,0,0.210526,0.0,0.131579,0.131579,remainder of shutoffs will continue until noon...



Sentence for top_k = 0
('remainder of shutoffs will continue until noon tomorrow .<n>The aim of the '
 'shutoffs is to reduce the risk of wildfires .')


Testing temperature from 0.2 to 2
- With top_k = 0
- And do_sample = True

In [35]:
# Updating hyperparameter to change
changed_hp = 'temperature'
range_start = 2
range_stop = 21
step = 2

# Getting the df
df_5 = loop_through_hp(cnnmodel, cnntokenizer, cnninputs["input_ids"],
  LONG_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=60,
  num_beams=1, early_stopping=False, no_repeat_ngram_size=0, do_sample=True,
  top_k=0, temperature=1, top_p=1)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_5)

100%|██████████| 10/10 [09:35<00:00, 57.50s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,2,0.294118,0.020000,0.156863,0.156863,The aim of the shutoffs is to reduce the risk ...
1,4,0.250000,0.039216,0.230769,0.230769,Nearly 800 thousand customers are scheduled to...
2,6,0.262626,0.061856,0.121212,0.121212,The aim of the blackouts is to reduce the risk...
3,8,0.349515,0.099010,0.213592,0.213592,These blackouts are due to high winds amid dry...
4,10,0.089552,0.030769,0.089552,0.089552,Pittsburgh based PG&E wanted eachMaharashtra r...
5,12,0.138614,0.000000,0.079208,0.079208,Nearly 800 davenably AGAINST McDonoughab retel...
6,14,0.040000,0.000000,0.040000,0.040000,Brakeloaded blackoutsrental will scams reservo...
7,16,0.000000,0.000000,0.000000,0.000000,тат Thousand believed Sierra Level Foam impact...
8,18,0.020408,0.000000,0.020408,0.020408,Norma 80% of ph cured Afghanchey crossed Minda...
9,20,0.000000,0.000000,0.000000,0.000000,Hide disempowerprevention pumpsarium Henan Kee...



Sentence for temperature = 2
('The aim of the shutoffs is to reduce the risk of wildfires .<n>The recent '
 'wave of precautionary shutoffs have drawn sharp criticism from Governor '
 'Gavin Newsom .<n>He blames PG&E for doing too little to properly maintain '
 'and secure its power lines against wind damage .')

Sentence for temperature = 4
('Nearly 800 thousand customers are scheduled to be affected by the shutoffs '
 'which are expected to last through at least midday tomorrow .<n>The aim is '
 'to reduce the risk of wildfires .<n>It would be the fourth round of mass '
 'blackouts imposed by the utility since Oct. 9 .')

Sentence for temperature = 6
('The aim of the blackouts is to reduce the risk of wildfires .<n>It is the '
 'fourth round of mass blackouts imposed by the utility since October 9 '
 '.<n>The record breaking drought has made the current conditions even worse '
 'than in previous years .')

Sentence for temperature = 8
('These blackouts are due to high winds amid dry

Testing top_k from 5 to 50:
- With do_sample = True

In [36]:
# Updating hyperparameter to change
changed_hp = 'top_k'
range_start = 5
range_stop = 51
step = 5

# Getting the df
df_6 = loop_through_hp(cnnmodel, cnntokenizer, cnninputs["input_ids"],
  LONG_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=60,
  num_beams=1, early_stopping=False, no_repeat_ngram_size=0, do_sample=True,
  top_k=50, temperature=1, top_p=1)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_6)

100%|██████████| 10/10 [08:36<00:00, 51.68s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,5,0.371134,0.189474,0.268041,0.268041,The aim of the shutoff is to reduce the risk o...
1,10,0.392523,0.190476,0.336449,0.336449,"Nearly 800,000 customers are scheduled to be a..."
2,15,0.329897,0.000000,0.206186,0.206186,Power shut offs are to last until midday .<n>C...
3,20,0.234043,0.043478,0.191489,0.191489,Nearly 800 thousand customers are scheduled to...
4,25,0.259740,0.000000,0.181818,0.181818,The goal of the shutoffs is to reduce the risk...
5,30,0.211765,0.048193,0.117647,0.117647,Blackouts are scheduled to last until midday t...
6,35,0.250000,0.000000,0.145833,0.145833,The aim is to reduce the risk of wildfires .<n...
7,40,0.288660,0.042105,0.206186,0.206186,A total of nearly 800 thousand customers are s...
8,45,0.294118,0.020000,0.156863,0.156863,The aim of the shutoffs is to reduce the risk ...
9,50,0.285714,0.038835,0.171429,0.171429,The aim is to reduce the risk of wildfires.<n>...



Sentence for top_k = 5
('The aim of the shutoff is to reduce the risk of wildfires .<n> PG&E stated '
 'it schedules blackouts in response to forecasts for high wind .<n>The record '
 'breaking drought has made the current conditions even worse than in previous '
 'years .')

Sentence for top_k = 10
('Nearly 800,000 customers are scheduled to be affected by the shutoffs which '
 'are expected to last through at least midday tomorrow .<n>PG&E stated it '
 'scheduled the blackouts in response to forecasts for high winds amid dry '
 'conditions .<n>The aim is to reduce the risk of wildfires, which are')

Sentence for top_k = 15
('Power shut offs are to last until midday .<n>Comes amid a historic drought '
 'in California .<n>The aim of the shutting downs is to reduce the risk of '
 'wildfires .<n>Governor Gavin Newsom criticised the shutting offs as being '
 'too broad .')

Sentence for top_k = 20
('Nearly 800 thousand customers are scheduled to be affected by the shutoffs '
 'which are 

Testing top_p from 0.1 to 0.9:
- With do_sample = True
- And top_k = 0

In [37]:
# Updating hyperparameter to change
changed_hp = 'top_p'
range_start = 10
range_stop = 100
step = 10

# Getting the df
df_7 = loop_through_hp(cnnmodel, cnntokenizer, cnninputs["input_ids"],
  LONG_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=60,
  num_beams=1, early_stopping=False, no_repeat_ngram_size=0, do_sample=True,
  top_k=0, temperature=1, top_p=1)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_7)

100%|██████████| 9/9 [08:56<00:00, 59.65s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,10,0.306122,0.062500,0.163265,0.163265,The aim of the shutoffs is to reduce the risk ...
1,20,0.306122,0.062500,0.163265,0.163265,The aim of the shutoffs is to reduce the risk ...
2,30,0.242424,0.000000,0.161616,0.161616,The aim of the shutoffs is to reduce the risk ...
3,40,0.242424,0.000000,0.161616,0.161616,The aim of the shutoffs is to reduce the risk ...
4,50,0.288462,0.078431,0.173077,0.173077,Power shutoffs are expected to last through at...
5,60,0.415842,0.222222,0.316832,0.316832,The aim of the shutoffs is to reduce the risk ...
6,70,0.427184,0.217822,0.349515,0.349515,PG&E announced shutoffs to save power during h...
7,80,0.326531,0.125000,0.204082,0.204082,If PG&E goes through with another public safet...
8,90,0.384615,0.176471,0.288462,0.288462,The aim of the power cuts is to reduce the ris...



Sentence for top_p = 10
('The aim of the shutoffs is to reduce the risk of wildfires .<n>The recent '
 'wave of precautionary shutoffs have drawn sharp criticism from Governor '
 'Gavin Newsom .<n>The record breaking drought has made the current conditions '
 'even worse than in previous years .')

Sentence for top_p = 20
('The aim of the shutoffs is to reduce the risk of wildfires .<n>The recent '
 'wave of precautionary shutoffs have drawn sharp criticism from Governor '
 'Gavin Newsom .<n>The record breaking drought has made the current conditions '
 'even worse than in previous years .')

Sentence for top_p = 30
('The aim of the shutoffs is to reduce the risk of wildfires .<n>It would be '
 'the fourth round of mass blackouts imposed by the utility since Oct. 9 '
 '.<n>The recent wave of precautionary shutoffs have drawn sharp criticism '
 'from Governor Gavin Newsom .')

Sentence for top_p = 40
('The aim of the shutoffs is to reduce the risk of wildfires .<n>It would be '
 'the f

Trying best sampling values:
- With do_sample = True
- And top_p = 0.7
- And top_k = 10
- And temperature = 0.8

In [39]:
# Updating hyperparameter to change
changed_hp = 'top_k'
range_start = 10
range_stop = 11
step = 1

# Getting the df
df_8 = loop_through_hp(cnnmodel, cnntokenizer, cnninputs["input_ids"],
  LONG_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=100,
  num_beams=1, early_stopping=False, no_repeat_ngram_size=0, do_sample=True,
  top_k=10, temperature=0.8, top_p=0.7)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_8)

100%|██████████| 1/1 [01:02<00:00, 62.89s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,10,0.294118,0.02,0.156863,0.156863,The aim of the shutoffs is to reduce the risk ...



Sentence for top_k = 10
('The aim of the shutoffs is to reduce the risk of wildfires .<n>The recent '
 'wave of precautionary shutoffs have drawn sharp criticism from Governor '
 'Gavin Newsom .<n>Newsom blames PG&E for doing too little to properly '
 'maintain and secure its power lines against wind damage .')


Testing min_lengths from 10 to 60:
- With do_sample = True
- And top_p = 0.7
- And top_k = 10
- And temperature = 0.8

In [40]:
# Updating hyperparameter to change
changed_hp = 'min_length'
range_start = 10
range_stop = 61
step = 10

# Getting the df
df_9 = loop_through_hp(cnnmodel, cnntokenizer, cnninputs["input_ids"],
  LONG_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=100,
  num_beams=1, early_stopping=False, no_repeat_ngram_size=0, do_sample=True,
  top_k=10, temperature=0.8, top_p=0.7)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_9)

100%|██████████| 6/6 [06:11<00:00, 61.91s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,10,0.238095,0.000000,0.166667,0.166667,The aim of the shutoffs is to reduce the risk ...
1,20,0.406780,0.086207,0.220339,0.220339,The aim is to reduce the risk of wildfires .<n...
2,30,0.421053,0.172043,0.357895,0.357895,PG&E scheduled the blackouts in response to fo...
3,40,0.242424,0.000000,0.141414,0.141414,The aim of the blackouts is to reduce the risk...
4,50,0.407407,0.188679,0.314815,0.314815,PG&E scheduled the blackouts in response to fo...
5,60,0.272727,0.074074,0.181818,0.181818,PG&E is scheduled to turn off power to nearly ...



Sentence for min_length = 10
('The aim of the shutoffs is to reduce the risk of wildfires .<n>It would be '
 'the fourth round of mass blackouts imposed by the utility since Oct. 9 .')

Sentence for min_length = 20
('The aim is to reduce the risk of wildfires .<n>If PG&E goes through with '
 'another public safety power shutoff, it would be the fourth round of mass '
 'blackouts imposed by the utility since Oct. 9 .<n>The recent wave of '
 'precautionary shutoffs have drawn sharp criticism from Governor Gavin '
 'Newsom, state regulators and consumer activists as being overly broad in '
 'scale .')

Sentence for min_length = 30
('PG&E scheduled the blackouts in response to forecasts for high winds amid '
 'dry conditions .<n>The aim is to reduce the risk of wildfires .<n>The recent '
 'wave of precautionary shutoffs have drawn sharp criticism from Governor '
 'Gavin Newsom .')

Sentence for min_length = 40
('The aim of the blackouts is to reduce the risk of wildfires .<n>It would be '

Comparing efficacy against best beam search no repeat ngram parameters:
- With early_stopping = True
- And num_beams = 4
- And no_repeat_ngram_size = 3

In [41]:
# Updating hyperparameter to change
changed_hp = 'no_repeat_ngram_size'
range_start = 3
range_stop = 4
step = 1

# Getting the df
df_10 = loop_through_hp(cnnmodel, cnntokenizer, cnninputs["input_ids"],
  LONG_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=60,
  num_beams=4, early_stopping=True, no_repeat_ngram_size=3, do_sample=False,
  top_k=50, temperature=1, top_p=1)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_10)

100%|██████████| 1/1 [01:12<00:00, 72.12s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,3,0.378947,0.215054,0.294737,0.294737,Nearly 800 thousand customers are scheduled to...



Sentence for no_repeat_ngram_size = 3
('Nearly 800 thousand customers are scheduled to be affected by the shutoffs '
 'which are expected to last through at least midday tomorrow .<n>PG&E stated '
 'it scheduled the blackouts in response to forecasts for high winds amid dry '
 'conditions .')


Best candidate and its respective Rouge-L score:
- max_new_tokens = 60
- min_length = 30
- top_k = 10
- temperature = 0.8
- top_p = 0.7

In [44]:
# Updating hyperparameter to change
changed_hp = 'min_length'
range_start = 30
range_stop = 31
step = 10

# Getting the df
df_11 = loop_through_hp(cnnmodel, cnntokenizer, cnninputs["input_ids"],
  LONG_REFERENCE, changed_hp, range_start, range_stop, step, max_new_tokens=60,
  num_beams=1, early_stopping=False, no_repeat_ngram_size=0, do_sample=True,
  top_k=10, temperature=0.8, top_p=0.7)

# Displaying the df and pretty printing the sentences
display_df_and_sent(df_11)

100%|██████████| 1/1 [00:55<00:00, 55.51s/it]


,hp_value,rouge1,rouge2,rougeL,rougeLsum,sent_gen
0,30,0.412371,0.168421,0.350515,0.350515,PG&E stated it scheduled the blackouts in resp...



Sentence for min_length = 30
('PG&E stated it scheduled the blackouts in response to forecasts for high '
 'winds amid dry conditions .<n>The aim is to reduce the risk of wildfires '
 '.<n>The recent wave of precautionary shutoffs have drawn sharp criticism '
 'from Governor Gavin Newsom .')


Okay, you're done.

Which model do you think produced the best summaries keeping in mind that best is in the eye of the reader?
- The latter two both did a good job. In terms of pegasus for longer generation, it appears that the combination of min and max length, top_k and top_p sampling, along with temperature adjustments, provided the best results.